## 2_filter_and_extract_buildings_data-Kenya

This notebook was executed to extract buildings for Nairobi along with the following attributes: 
- longitude
- latitude
- ID (longitude:latitude)
- bf source
- confidence
- area_in_meters
- perimeter_in_meters
- building faces
- geometry


It begins by defining the IBM COS configuration parameters, importing the required libraries, and initializing the COS client. An external utility `filtering_grid_generator.py` is downloaded from IBM's UTILS_BUCKET. The notebook then reads the Nairobi boundary GeoJSON, and loads the Kenya VIDA buildings GeoParquet (downloaded from the first notebook). After reprojecting the data to EPSG:4326, it computes centroid coordinates and constructs a unique ID for each building. A grid-based filtering method is applied: grid tiles over the Nairobi polygon allow fast pre-selection using tile bounding boxes, and an exact geometric `contains()` test is used only for buildings near the edges. All buildings determined to lie within the Nairobi boundary are merged, along with their attributes, after which the dataset is filtered to retain only those whose VIDA confidence exceed a threshold of 0.7. The final filtered buildings are then saved as a Parquet file.

In [ ]:
# Read notebook configuration
import getpass
import json

#config_str = getpass.getpass('Enter your prepared config: ')
config_str = '''

{
"COUNTRY_NAME": "Kenya",
"OUTPUT_BUCKET": "",
"VIDA_COUNTRIES_BUILDINGS": "",

"COS_ENDPOINT_URL": "https://s3.direct.eu-de.cloud-object-storage.appdomain.cloud",
"COS_AUTH_ENDPOINT_URL": "",
"COS_APIKEY": "",
"UTILS_BUCKET": ""
}

'''

config = json.loads(config_str)


In [8]:
# Import necessary libraries
import pyarrow.parquet as pq
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely
import time
import threading
import datetime 
import jaydebeapi as jdbc
import jpype
import os
from tqdm import tqdm
from botocore.client import Config
import ibm_boto3
import io
import time
import pickle

from pyproj import Geod

geod = Geod(ellps="WGS84")

In [9]:
miniminal_vida_confidence = 0.7

In [10]:
# init S3 client in order to work with last tiff file version
cos_client = ibm_boto3.client(service_name='s3',
                                  ibm_api_key_id=config["COS_APIKEY"],
                                  ibm_auth_endpoint="https://iam.cloud.ibm.com/oidc/token",
                                  config=Config(signature_version='oauth'),
                                  endpoint_url=config["COS_ENDPOINT_URL"])


# import external utils library
response = cos_client.list_objects_v2(Bucket=config["UTILS_BUCKET"])

utils_to_download = ['filtering_grid_generator.py'] 

try:
    for obj in response['Contents']:
        name = obj['Key']
        if name in utils_to_download:
            streaming_body_1 = cos_client.get_object(Bucket=config["UTILS_BUCKET"], Key=name)['Body']
            print("Copying to localStorage :  " + name)
            with io.FileIO(name, 'w') as file:
                for i in io.BytesIO(streaming_body_1.read()):
                    file.write(i)
                    
    from filtering_grid_generator import GridGenerator
    
    print('External utils succesfully imported')
except Exception as e:
    print('Error occured: ', e)


Copying to localStorage :  filtering_grid_generator.py
External utils succesfully imported


In [5]:
# Define input
country_parquet = r"parquets/KEN.parquet"
AOI_boundary = "osm_boundary_nairobi.geojson"

# Laod Nairobi boundary polygon
nairobi_gdf = gpd.read_file(AOI_boundary)
nairobi_gdf = nairobi_gdf.to_crs("EPSG:4326")

# Extract geometry
geom = nairobi_gdf.geometry.iloc[0]

# Split multipolygon into polygon list
if geom.geom_type == "MultiPolygon":
    polygons = list(geom.geoms)
else:
    polygons = [geom]

# Correct final structure
regions_polygons = {
    "Nairobi": polygons
}

regions_polygons


{'Nairobi': [<POLYGON ((36.68 -1.33, 36.683 -1.337, 36.685 -1.337, 36.685 -1.337, 36.686 ...>]}

In [ ]:
# Load entire country buildings 
df = gpd.read_parquet(country_parquet)
df = df.to_crs("EPSG:4326")

print(f"Total buildings in {country_parquet} is: {len(df):,}")
print("Computing centroids (longitude, latitude)...")

# Apply centroid computation using tqdm progress bar
tqdm.pandas(desc="Computing longitude")
df["longitude"] = df["geometry"].progress_apply(lambda g: g.centroid.xy[0][0])

tqdm.pandas(desc="Computing latitude")
df["latitude"] = df["geometry"].progress_apply(lambda g: g.centroid.xy[1][0])

tqdm.pandas(desc="Computing ID")
df["id"] = (df["longitude"].progress_apply(str) + ":" + df["latitude"].progress_apply(str))

print("Centroid + ID computation completed.")
list(df.columns)

Total buildings in parquets/KEN.parquet is: 27,916,511
Computing centroids (longitude, latitude)...


Computing longitude:  10%|█         | 2833059/27916511 [00:50<07:24, 56424.30it/s]

In [ ]:
# Generate grids
grids = GridGenerator()

for region_name, region_polygons in regions_polygons.items():
    print('\033[96mProcessing region', region_name)

    for pidx, region_polygon in enumerate(region_polygons):
        print('\033[93m    processing pidx', pidx)

        inside_polygon_grid, outside_polygon_grid = grids.generate_grids(region_polygon, n=150)

        # Fast filtering (INSIDE tiles)
        print('\033[93m    Filtering buildings inside polygon tiles')
        buildings_in_tiles = []
        buildings_found = 0

        for in_poly_tile in tqdm(inside_polygon_grid, total=len(inside_polygon_grid), desc='Filtering buildings inside polygon'):
            min_lon, min_lat, max_lon, max_lat = in_poly_tile.bounds

            buildings_in_tile = df[
                    (df.longitude >= min_lon) &
                    (df.longitude <= max_lon) &
                    (df.latitude >= min_lat) &
                    (df.latitude <= max_lat)
            ]

            if len(buildings_in_tile) > 0:
                buildings_in_tiles.append(buildings_in_tile)
                buildings_found += len(buildings_in_tile)

        try:
            buildings_in_tiles_df = pd.concat(buildings_in_tiles)
        except:
            buildings_in_tiles_df = pd.DataFrame()

        # Fast filtering (OUTSIDE tiles)
        print('\033[93m    Filtering buildings outside polygon tiles')
        buildings_out_tiles = []
        buildings_out_tiles_found = 0

        for off_poly_tile in tqdm(outside_polygon_grid, total=len(outside_polygon_grid), desc='Filtering buildings outside polygon'):
            min_lon, min_lat, max_lon, max_lat = off_poly_tile.bounds

            buildings_out_tile = df[
                    (df.longitude >= min_lon) &
                    (df.longitude <= max_lon) &
                    (df.latitude >= min_lat) &
                    (df.latitude <= max_lat)
            ]

            if len(buildings_out_tile) > 0:
                buildings_out_tiles.append(buildings_out_tile)
                buildings_out_tiles_found += len(buildings_out_tile)
                
        try:
            buildings_out_tiles_df = pd.concat(buildings_out_tiles)
        except:
            buildings_out_tiles_df = pd.DataFrame()

        # AMBIGUOUS BUILDINGS (near boundary)
        # Case 1: Both inside and outside exist
        if (len(buildings_in_tiles_df) != 0) and (len(buildings_out_tiles_df) != 0):
            try:
                print(f'\033[93m        Buildings found in polygon tiles {len(buildings_in_tiles_df)}, outside tiles {len(buildings_out_tiles_df)} remaining buildings {len(df) - len(buildings_in_tiles_df) - len(buildings_out_tiles_df)}')

                # remove inside
                if len(buildings_in_tiles_df) > 0:
                    remaining_buildings_df = df[~df['id'].isin(buildings_in_tiles_df['id'])]

                    # remove outside
                    if len(buildings_out_tiles_df) > 0:
                        remaining_buildings_df = remaining_buildings_df[~remaining_buildings_df['id'].isin(buildings_out_tiles_df['id'])]

                # Case 2: inside empty, outside non-empty
                if (len(buildings_in_tiles_df) == 0) and (len(buildings_out_tiles_df) > 0):
                    remaining_buildings_df = df[~df['id'].isin(buildings_out_tiles_df['id'])]

                print(f'\033[93m        Remaining amount check {len(remaining_buildings_df)}')

                result_df = pd.DataFrame()

                # GEOMETRIC TEST ON REMAINING
                if len(remaining_buildings_df) > 0:
                    remaining_buildings_df['buildings_in_polygon'] = [region_polygon.contains(shapely.Point(row.longitude, row.latitude))
                        for row in tqdm(remaining_buildings_df.itertuples(), total=len(remaining_buildings_df), desc='Filtering buildings')
                    ]

                    remaining_buildings_df = remaining_buildings_df[remaining_buildings_df.buildings_in_polygon == True]

                    # Merge inside + remaining
                    if len(buildings_in_tiles_df) > 0:
                        if len(remaining_buildings_df) > 0:
                            result_df = pd.concat([buildings_in_tiles_df, remaining_buildings_df])
                        else:
                            result_df = buildings_in_tiles_df
                    else:
                        if len(remaining_buildings_df) > 0:
                            result_df = remaining_buildings_df
                        else:
                            result_df = pd.DataFrame()
                else:
                    if len(buildings_in_tiles_df) > 0:
                        result_df = buildings_in_tiles_df
                    else:
                        result_df = pd.DataFrame()

                # save intermediate result
                if len(result_df) > 0:
                    print(f'\033[93m        Filtered buildings: {len(result_df)}')
                    result_df.to_parquet(f"{region_name}_raw_filtered_buildings.parquet")

            except Exception as e:
                print("Error:", e)

  


In [ ]:
def get_inner_faces(interiors):
    total = 0
    for interior in interiors:
        total += len(interior.coords) - 1
    return total

def get_inner_perimeter(interiors):
    geod = Geod(ellps="WGS84")
    total = 0
    for interior in interiors:
        total += abs(geod.geometry_area_perimeter(interior)[1])
    return total


def calculations(buildings_in_polygon):
    
    # filter out google buildings vith cobfidence < miniminal_vida_confidence
    google_buildings = buildings_in_polygon[(buildings_in_polygon.bf_source == 'google') & (buildings_in_polygon.confidence > miniminal_vida_confidence)]
    microsoft_buildings = buildings_in_polygon[buildings_in_polygon.bf_source == 'microsoft']
    
    buildings_in_polygon = pd.concat([google_buildings, microsoft_buildings])
    
    del google_buildings
    del microsoft_buildings

    # Compute area
    buildings_in_polygon['area_in_meters'] = buildings_in_polygon["geometry"].apply(lambda g: abs(geod.geometry_area_perimeter(g)[0]))

    # Compute Perimeter
    buildings_in_polygon.insert(0,"perimeter_in_meters",0)
    buildings_in_polygon['perimeter_in_meters'] = buildings_in_polygon["geometry"].apply(lambda g: (abs(geod.geometry_area_perimeter(g)[1]) + get_inner_perimeter(g.interiors)) if (g.geom_type == "Polygon") else (abs(geod.geometry_area_perimeter(g)[1])))

    # Compute number of faces
    buildings_in_polygon.insert(1,"building_faces",0)
    buildings_in_polygon['building_faces'] = buildings_in_polygon["geometry"].apply(lambda g: (len(g.exterior.coords) - 1 + get_inner_faces(g.interiors)) if (g.geom_type == "Polygon") else 0)

    
    return buildings_in_polygon

In [ ]:
regions_polygons.keys()

In [ ]:

columns = ['bf_source', 'confidence', 'geometry', 'longitude', 'latitude', 'id']

for region_name in regions_polygons.keys():
    region_name = region_name.replace(" ", "_")

    try:
        # LOAD THE REGION'S RAW FILTERED PARQUET
        raw_path = f"{region_name}_raw_filtered_buildings.parquet"

        print(f"\033[96mLoading raw filtered buildings for {region_name}...")
        main_df = gpd.read_parquet(raw_path, columns=columns)

        # DROP DUPLICATES 
        main_df = main_df.drop_duplicates(subset=['id'])

        # RUN GEOMETRY / VIDA CALCULATIONS
        main_df = calculations(main_df)

        # SAVE FINAL REGION BUILDINGS PARQUET
        filename = f"{region_name}_buildings.parquet"
        main_df.to_parquet(filename)
        print(f"{filename} saved successfully with {len(main_df)} buildings")

        # OPTIONAL: UPLOAD TO BUCKET 
        try:
            res = cos_client.upload_file(Filename=filename, Bucket=config["OUTPUT_BUCKET"], Key=filename)
        except Exception as e:
            print("Error uploading:", e)
        else:
            print(f"{filename} successfully uploaded")

    except Exception as e:
        print(f"Region calculations error for {region_name}: {e}")


In [ ]:
print(list(main_df.columns))